# 2장 실습 — 도로망 데이터 열어 보기

시뮬레이터가 달리는 무대를 직접 엽니다. 교재 2장 전체에 대응합니다.

이 장의 핵심은 하나입니다. 도로망은 특별한 자료구조가 아니라 표 두 개입니다.
노드 표와 엣지 표를 읽고, 인접 리스트로 바꾸고, 좌표를 붙이면 최단경로를 구할 준비가 끝납니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 표 두 개 (교재 2.1)

In [ ]:
import pandas as pd

from smartmob.data import data_path

nodes = pd.read_parquet(data_path("hanam/road_graph_nodes.parquet"))
edges = pd.read_parquet(data_path("hanam/road_graph_edges.parquet"))

print("노드 표", nodes.shape)
print("엣지 표", edges.shape)
edges[["edge_id", "highway", "name", "length", "free_flow_speed_kmh", "oneway"]].head()

엣지 하나가 교차로와 교차로 사이의 도로 한 토막입니다.
`length` 는 미터, `free_flow_speed_kmh` 는 막히지 않을 때의 속도입니다.

## 2. 도로 종류별 개수 (교재 2.3)

엣지의 절반 가까이는 자동차가 다닐 수 없는 길입니다.

In [ ]:
top = edges["highway"].value_counts().head(10)
top

가장 많은 것이 `footway`, 즉 보도입니다.
택시 시뮬레이션에 보도를 넣으면 차가 인도로 달립니다. 그래서 걸러 내야 합니다.

In [ ]:
from smartmob.data import load_road_graph

drive = load_road_graph("hanam", modes=("drive",))
walkable = load_road_graph("hanam", modes=("drive", "walk", "bike"))

banner("modes 필터의 효과")
expect("자동차 도로 노드", drive.n_nodes, 12_566)
expect("자동차 도로 엣지", drive.n_edges, 28_589)
print(f"전체 노드 {walkable.n_nodes:,}, 전체 엣지 {walkable.n_edges:,}")
print(f"자동차가 쓸 수 있는 엣지는 전체의 {drive.n_edges / walkable.n_edges:.0%} 입니다")

## 3. 인접 리스트 (교재 2.4)

표를 그대로 두면 "이 노드에서 갈 수 있는 곳"을 찾을 때마다 표 전체를 훑어야 합니다.
미리 노드마다 이웃 목록을 만들어 두는 것이 인접 리스트입니다.
`RoadGraph` 가 읽을 때 이미 만들어 둡니다.

In [ ]:
sample = next(n for n in drive.nodes if len(drive.adj[n]) >= 3)

banner(f"노드 {sample} 의 이웃")
for neighbour, seconds, edge_index in drive.neighbors(sample):
    print(f"  → {neighbour}  {seconds:6.1f}초  (엣지 {edge_index})")

값이 거리가 아니라 **초**인 것이 중요합니다.
최단경로를 거리로 풀면 좁은 골목이 뽑히고, 시간으로 풀면 큰길이 뽑힙니다.
시뮬레이터가 알고 싶은 것은 시간입니다.

## 4. 좌표로 노드 찾기 (교재 2.5)

승객은 위경도로 호출합니다. 그 위치를 가장 가까운 노드에 붙이는 것을 스냅이라고 합니다.

In [ ]:
HANAM_CITY_HALL = (37.5393, 127.2148)
MISA_STATION = (37.5606, 127.1926)

start = drive.nearest_node(*HANAM_CITY_HALL)
goal = drive.nearest_node(*MISA_STATION)

print("하남시청 →", start, drive.coord[start])
print("미사역   →", goal, drive.coord[goal])

In [ ]:
from smartmob.teaching.graph import haversine_km

print(f"두 지점 직선거리 {haversine_km(*HANAM_CITY_HALL, *MISA_STATION):.2f} km")

## 5. 빈칸

### 5.1 가장 긴 엣지

자동차 도로 중 `length` 가 가장 긴 엣지 다섯 개를 찾고, 그 도로의 이름과 종류를 봅니다.
왜 이런 엣지가 길게 나오는지 한 줄로 적습니다.

In [ ]:
drive_edges = drive.edges       # modes 필터를 통과한 엣지만 남은 표
print("자동차 도로 엣지 표", drive_edges.shape)

longest = None      # drive_edges 에서 length 상위 5개 행 (DataFrame)

banner("빈칸 5.1")
todo("가장 긴 엣지 5개", longest, fmt=lambda df: f"{len(df)}행, 최장 {df['length'].max():.0f}m")

### 5.2 속도가 0인 엣지

`free_flow_speed_kmh` 가 0이거나 비어 있는 엣지가 있으면 소요시간이 무한대가 됩니다.
자동차 도로망에 그런 엣지가 몇 개 있는지 셉니다.

In [ ]:
bad_speed = None    # 속도가 0이거나 결측인 자동차 도로 엣지 수 (정수)

banner("빈칸 5.2")
todo("속도가 없는 엣지", bad_speed)

### 5.3 한 방향만 뚫린 길

`oneway` 가 참인 엣지의 비율을 구합니다.
일방통행을 무시하고 최단경로를 구하면 무엇이 잘못되는지 한 줄로 적습니다.

In [ ]:
oneway_share = None     # 일방통행 엣지의 비율 (0~1)

banner("빈칸 5.3")
todo("일방통행 비율", oneway_share, fmt=lambda v: f"{v:.1%}")

## 정리

- 도로망은 노드 표와 엣지 표입니다. `RoadGraph` 가 둘을 읽어 인접 리스트로 바꿉니다
- `modes` 로 거르지 않으면 택시가 보도로 달립니다
- 엣지의 비용은 거리가 아니라 초입니다
- `nearest_node` 가 위경도를 노드에 붙입니다
- 3장 실습에서는 이 인접 리스트 위에서 최단경로를 직접 짭니다